# SAT/SMT for ML

## What This Is
SMT solvers can encode ML problems as constraint satisfaction queries. Key applications:

1. **Adversarial example finding**: Is there an input within an ε-ball of x that changes the classification?
2. **Monotonicity verification**: Does the network respect known domain constraints (e.g., higher income → higher credit score)?
3. **Fairness verification**: Can the protected attribute influence the output?
4. **Reachability analysis**: Can the output ever enter a specific dangerous region?

## Encoding Neural Networks as SMT
For a ReLU network, each neuron has two cases:
- If pre-activation ≤ 0: output = 0 (linear, inactive)
- If pre-activation > 0: output = pre-activation (linear, active)

This creates a piecewise-linear structure that SMT solvers can handle by reasoning over which neurons are active vs inactive.

In [ ]:
import numpy as np
from typing import Tuple, List, Optional

# SMT encoding of a tiny neural network
# We manually implement the core idea: enumerate activation patterns

try:
    from z3 import *
    HAS_Z3 = True
except ImportError:
    HAS_Z3 = False

np.random.seed(42)

# Simple 2-input, 2-hidden, 1-output ReLU network
W1 = np.array([[1.5, -1.0], [-1.5, 2.0]])
b1 = np.array([0.5, -0.5])
W2 = np.array([[1.0, 1.0]])
b2 = np.array([0.0])

def net_forward(x):
    h = np.maximum(0, W1 @ x + b1)
    return (W2 @ h + b2)[0]

def net_classify(x):
    return 1 if net_forward(x) > 0 else 0

x0 = np.array([1.0, 0.5])
print(f'Network output at x0: {net_forward(x0):.4f}')
print(f'Classification: {net_classify(x0)}')


In [ ]:
if HAS_Z3:
    # Find adversarial example within L-inf ball of x0
    # Is there x in [x0-eps, x0+eps] such that net(x) <= 0?
    
    eps = 0.3
    x1, x2 = Reals('x1 x2')
    h1, h2 = Reals('h1 h2')  # hidden neurons
    pre1, pre2 = Reals('pre1 pre2')  # pre-activations
    y = Real('y')  # output
    
    s = Solver()
    
    # Input constraints: x in [x0 - eps, x0 + eps]
    s.add(x1 >= x0[0] - eps, x1 <= x0[0] + eps)
    s.add(x2 >= x0[1] - eps, x2 <= x0[1] + eps)
    
    # Layer 1 pre-activations
    s.add(pre1 == 1.5*x1 - 1.0*x2 + 0.5)
    s.add(pre2 == -1.5*x1 + 2.0*x2 - 0.5)
    
    # ReLU activation (case split)
    s.add(h1 == If(pre1 >= 0, pre1, RealVal(0)))
    s.add(h2 == If(pre2 >= 0, pre2, RealVal(0)))
    
    # Output
    s.add(y == h1 + h2)
    
    # Adversarial objective: find input where output flips (y <= 0)
    s.add(y <= 0)
    
    result = s.check()
    print(f'Adversarial example search (eps={eps}):')
    print(f'  Result: {result}')
    
    if result == sat:
        m = s.model()
        x_adv = np.array([float(m[x1].as_decimal(5)), float(m[x2].as_decimal(5))])
        print(f'  Adversarial input found: {x_adv}')
        print(f'  Network output: {net_forward(x_adv):.4f} (should be <=0)')
    else:
        print(f'  No adversarial example exists in eps={eps} ball: CERTIFIED ROBUST')
else:
    # Enumerate adversarial candidates manually (brute force)
    eps = 0.3
    adv_found = None
    for _ in range(10000):
        x_cand = x0 + np.random.uniform(-eps, eps, 2)
        if net_classify(x_cand) != net_classify(x0):
            adv_found = x_cand
            break
    if adv_found is not None:
        print(f'Adversarial example found at: {adv_found}')
        print(f'Output: {net_forward(adv_found):.4f}')
    else:
        print(f'No adversarial example found (empirical, not certified)')
